# HPO SolarSystemLander

Elise HPO (8D) from `hpo/designs/design.md`.

## Set up

### Set up Colab

In [ ]:
# cell: colab-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.notebook.colab import setup_colab

COLAB = setup_colab()

### Set up HPO

In [ ]:
# cell: hpo-setup; requires: colab-setup

import torch
from hpo.notebook.colab import prepare_storage

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OBSERVATION_MODE = "10d"
DATABASE = f"solar_system_lander_{OBSERVATION_MODE}_elise_accel"
STORAGE = prepare_storage(COLAB, DATABASE)

## Optimize HPs iteratively

### Define baseline

Choose on of the following alternatives.

#### For a new study series

In [ ]:
# cell: baseline-1

from hpo.hyperparams import HP
from hpo.study import Baseline

_BASELINE_PARAMS = {
    HP.LEARNING_RATE: 0.0022727854024196057,
    HP.BATCH_SIZE: 512,
    HP.EPS_END: 0.047716002108220544,
    HP.EPS_DECAY: 43_214,
    HP.GAMMA: 0.99,
    HP.TAU: 0.005,
    HP.LEARNING_STARTS: 2_500,
    HP.OPTIMIZE_EVERY: 2,
    HP.REPLAY_MEMORY: 400_000,
    HP.NUM_EPISODES: 500,
}

baseline = Baseline(params=_BASELINE_PARAMS)

#### For a new study in a series

In [ ]:
# cell: baseline-2

_STUDY_NAME = ...

import optuna

from hpo.notebook.optuna import db_path
from hpo.study import Baseline

_db_path = db_path(DATABASE, google_drive=True)

_study = optuna.load_study(
    study_name=_STUDY_NAME,
    storage=f"sqlite:///{_db_path}",
)

_best_trial = max(
    [trial for trial in _study.trials if trial.value is not None],
    key=lambda trial: trial.value,
)

baseline = Baseline(
    params=_best_trial.params,
    score=_best_trial.value,
)

print(_best_trial.number, _best_trial.value)
baseline.params

#### For continuation after Colab runtime loss

In [ ]:
# cell: baseline-2

import optuna

from hpo.notebook.optuna import db_path
from hpo.study import Baseline

_STUDY_NAME = "s3_10d_better_space"

import optuna

from hpo.study import Baseline

_db_path = db_path(DATABASE, google_drive=True)

_study = optuna.load_study(
    study_name=_STUDY_NAME,
    storage=f"sqlite:///{_db_path}",
)

_best_trial = max(
    [trial for trial in _study.trials if trial.value is not None],
    key=lambda trial: trial.value,
)

baseline = Baseline(
    params=_best_trial.params,
    score=_study.user_attrs.get("checkpoint_min_score"),
)

baseline.params

### Create study runner

In [ ]:
# cell: environment-factory; requires: hpo-setup

from hpo.solar_system_lander.environment import EnvFactory, World
from hpo.solar_system_lander.reward_shaping import GroundThrustPenaltyEnv

REWARD_SHAPING = None

# REWARD_SHAPING = {
#     "name": "ground_thrust_penalty",
#     "ground_thrust_penalty": 0.5,
# }

TRAINING_ENV_WRAPPER = (
    None
    if REWARD_SHAPING is None
    else lambda env: GroundThrustPenaltyEnv(env, ground_thrust_penalty=REWARD_SHAPING["ground_thrust_penalty"])
)

ENV_FACTORY = EnvFactory(
    OBSERVATION_MODE,
    world_mix={
        World.MERCURY: 1,
        World.VENUS: 4,
        World.EARTH: 4,
        World.MOON: 1,
        World.MARS: 1,
    },
    training_env_wrapper=TRAINING_ENV_WRAPPER,
)

STUDY_ATTRS = ENV_FACTORY.metadata()
if REWARD_SHAPING is not None:
    STUDY_ATTRS["reward_shaping"] = REWARD_SHAPING


In [ ]:
# cell: objective-config; requires: colab-setup, hpo-setup, environment-factory

from hpo.checkpointing import ObjectiveHookFactory
from hpo.objective import ObjectiveConfig

_EARLY_STOPPING_SCORE = -250.0  # optional; default: -250.0; disable: None
OBJECTIVE_CFG = ObjectiveConfig(
    environment_factory=ENV_FACTORY,
    num_envs=22,
    device=device,
    eval_episodes=10,
    early_stopping_score=_EARLY_STOPPING_SCORE,
    hooks=ObjectiveHookFactory(
        checkpoint_dir=COLAB.local_study_dir / f"{DATABASE}_checkpoints",
        best_eval_archive_dir=COLAB.drive_study_dir / "best_checkpoints" / DATABASE,
        window=100,
    ),
)


In [ ]:
# cell: study-runner; requires: hpo-setup, baseline, environment-factory, objective-config

from hpo.notebook.dashboard import Dashboard
from hpo.study import StudyRunner

_TRAINING_SCORE_MIN = -500.0  # optional; default: -500.0; disable: None

runner = StudyRunner(
    database_path=lambda _name: STORAGE.database_path,
    objective_cfg=OBJECTIVE_CFG,
    baseline=baseline,
    reporter=Dashboard(render_mode="safe", training_score_min=_TRAINING_SCORE_MIN),
    study_attrs=STUDY_ATTRS,
    robust_candidates=3,
    robust_eval_episodes=50,
    sync_fn=STORAGE.backup,
)


### Run studies

In [ ]:
# cell: run-s3-10d-better-space; requires: study-runner

from hpo.hyperparams import HP

def _suggest_params(trial, _incumbent_params):
    trial.suggest_categorical(HP.NUM_EPISODES, [2000])
    trial.suggest_int(HP.REPLAY_MEMORY, 30_000, 120_000, log=True)
    trial.suggest_float(HP.EPS_END, 0.025, 0.045)
    trial.suggest_int(HP.EPS_DECAY, 25_000, 80_000, log=True)
    trial.suggest_float(HP.LEARNING_RATE, 0.0005, 0.0012, log=True)
    trial.suggest_categorical(HP.BATCH_SIZE, [512])
    trial.suggest_categorical(HP.GAMMA, [0.99, 0.995])
    trial.suggest_categorical(HP.TAU, [0.002, 0.005])
    trial.suggest_categorical(HP.LEARNING_STARTS, [2_500, 5_000])
    trial.suggest_categorical(HP.OPTIMIZE_EVERY, [2])

runner.run("s3_10d_better_space", _suggest_params, 50)

In [ ]:
# cell: restore-253-checkpoint-local; requires: hpo-setup

from pathlib import Path
import shutil

_local = Path("/content/rl_lab/hpo/runs/solar_system_lander_10d_elise_accel_checkpoints/trials/trial_0035_eval_best.pt")
_archived = Path("/content/drive/MyDrive/rl_lab/hpo/best_checkpoints/solar_system_lander_10d_elise_accel/best_eval_checkpoint.pt")

_local.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(_archived, _local)

print("restored:", _local)
print("size:", _local.stat().st_size)

## Analyze

### Compute checkpoint scores

In [ ]:
# cell: checkpoint-world-distributions; requires: objective-config

from hpo.evaluation.checkpoint_robustness import checkpoint_scores, score_summary
from hpo.notebook.colab import path

_CHECKPOINT_PATH = path("best_checkpoints/solar_system_lander_10d_elise_accel/best_eval_checkpoint.pt", True)
_EPISODES = 200

scores = checkpoint_scores(_CHECKPOINT_PATH, OBJECTIVE_CFG, episodes=_EPISODES)
summary = score_summary(scores)

display(summary.round(1))

### Plot checkpoint scores

In [ ]:
# cell: heatmap-with-mean-median; requires: checkpoint-world-distributions

from hpo.notebook import plots

plots.heatmap(scores, bins=200);

In [ ]:
# cell: quantiles; requires: checkpoint-world-distributions

from hpo.notebook import plots


plots.quantiles(scores);

### Select best checkpoint

In [ ]:
# cell: select-best-checkpoint; requires: hpo-setup

import optuna

from hpo.checkpointing import best_checkpoint

_STUDY_NAME = "(insert name)"
_study = optuna.load_study(
    study_name=_STUDY_NAME,
    storage=f"sqlite:///{STORAGE.database_path}",
)

best_checkpoint(_study)